# Modelagem para Business Intelligence

Nesta etapa, a base analítica é organizada para uso no Power BI, com definição das dimensões e dos campos necessários para construção dos indicadores e dashboards.

In [4]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"

base = pd.read_csv(
    PROCESSED / "base_analitica_escola_ano.csv",
    dtype={"codigo_escola": "string"}
)

base["ano"] = pd.to_numeric(
    base["ano"],
    errors="coerce"
).astype("Int64")


# Dimensão escola
dim_escola = (
    base[
        [
            "codigo_escola",
            "nome_escola",
            "dependencia_administrativa",
            "localizacao",
            "municipio",
            "uf",
        ]
    ]
    .drop_duplicates(subset=["codigo_escola"])
    .reset_index(drop=True)
)


# Dimensão tempo
dim_tempo = (
    base[["ano"]]
    .drop_duplicates()
    .sort_values("ano")
    .reset_index(drop=True)
)


print("Dimensão escola:", dim_escola.shape)
print("Dimensão tempo:", dim_tempo.shape)

display(dim_escola.head())
display(dim_tempo)

Dimensão escola: (276, 6)
Dimensão tempo: (6, 1)


,codigo_escola,nome_escola,dependencia_administrativa,localizacao,municipio,uf
0,43000479,E E IND ENS FUN TUPE PAN,Estadual,1,Porto Alegre,RS
1,43001254,E E IND ENS FUN PINDO POTY,Estadual,1,Porto Alegre,RS
2,43005047,E E IND ENS FUN KA AGUY MIRI,Estadual,1,Porto Alegre,RS
3,43048200,EMEF PORTO NOVO,Municipal,1,Porto Alegre,RS
4,43104932,COLEGIO DE APLICACAO UFRGS,Federal,1,Porto Alegre,RS


,ano
0,2018
1,2019
2,2020
3,2021
4,2022
5,2023


## Construção da tabela fato

A tabela fato concentra os indicadores educacionais e escolares na granularidade escola/ano, mantendo as chaves necessárias para relacionamento com as dimensões de escola e tempo.

In [5]:
colunas_fato = [
    "ano",
    "codigo_escola",

    # Rendimento
    "taxa_aprovacao",
    "taxa_reprovacao",
    "taxa_abandono",

    # Média de alunos
    "media_alunos_fund",
    "media_alunos_ai",
    "media_alunos_af",

    # Porte
    "QT_MAT_FUND",
    "QT_MAT_FUND_AI",
    "QT_MAT_FUND_AF",
    "QT_DOC_FUND",
    "QT_DOC_FUND_AI",
    "QT_DOC_FUND_AF",
    "QT_TUR_FUND",
    "QT_TUR_FUND_AI",
    "QT_TUR_FUND_AF",

    # Infraestrutura selecionada
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_BIBLIOTECA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_INTERNET",
    "IN_BANDA_LARGA",
    "IN_ACESSIBILIDADE_RAMPAS",
]

fato_desempenho = (
    base[colunas_fato]
    .copy()
    .sort_values(["ano", "codigo_escola"])
    .reset_index(drop=True)
)

print("Tabela fato:", fato_desempenho.shape)

print(
    "Chaves duplicadas:",
    fato_desempenho.duplicated(
        subset=["ano", "codigo_escola"]
    ).sum()
)

display(fato_desempenho.head())

Tabela fato: (1621, 23)
Chaves duplicadas: 0


,ano,codigo_escola,taxa_aprovacao,taxa_reprovacao,taxa_abandono,media_alunos_fund,media_alunos_ai,media_alunos_af,QT_MAT_FUND,QT_MAT_FUND_AI,...,QT_DOC_FUND_AF,QT_TUR_FUND,QT_TUR_FUND_AI,QT_TUR_FUND_AF,IN_ESGOTO_REDE_PUBLICA,IN_BIBLIOTECA,IN_LABORATORIO_INFORMATICA,IN_INTERNET,IN_BANDA_LARGA,IN_ACESSIBILIDADE_RAMPAS
0,2018,43000479,91.7,8.3,0.0,6.0,NaN,NaN,12,12,...,2,2,0,2,1,0,0,0,0.0,0
1,2018,43001254,100.0,0.0,0.0,1.5,1.0,NaN,3,3,...,1,2,1,1,0,0,0,0,0.0,0
2,2018,43005047,100.0,0.0,0.0,3.0,NaN,NaN,6,6,...,2,2,0,2,0,0,0,0,0.0,0
3,2018,43048200,78.0,22.0,0.0,26.9,26.0,29.0,350,234,...,8,13,9,4,1,1,1,1,1.0,0
4,2018,43104932,88.5,11.5,0.0,24.7,20.0,28.0,296,100,...,39,12,5,7,1,1,1,1,1.0,0


## Exportação do modelo para BI

As dimensões e a tabela fato são exportadas para arquivos separados, permitindo a construção do modelo de dados no Power BI.

In [6]:
BI = PROCESSED / "bi"
BI.mkdir(parents=True, exist_ok=True)

dim_escola.to_csv(
    BI / "dim_escola.csv",
    index=False
)

dim_tempo.to_csv(
    BI / "dim_tempo.csv",
    index=False
)

fato_desempenho.to_csv(
    BI / "fato_desempenho.csv",
    index=False
)

print("Arquivos exportados:")
print(BI / "dim_escola.csv")
print(BI / "dim_tempo.csv")
print(BI / "fato_desempenho.csv")

Arquivos exportados:
c:\Users\blanc\OneDrive\Documents\Projetos\Projeto-Em-Business-Intelligence-e-Analytics\data\processed\bi\dim_escola.csv
c:\Users\blanc\OneDrive\Documents\Projetos\Projeto-Em-Business-Intelligence-e-Analytics\data\processed\bi\dim_tempo.csv
c:\Users\blanc\OneDrive\Documents\Projetos\Projeto-Em-Business-Intelligence-e-Analytics\data\processed\bi\fato_desempenho.csv
